In [5]:
#### Remove images from Chats
import os
import shutil

source_dir = '/Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_chats/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages/inbox'

for item_name in os.listdir(source_dir):
    item_path = os.path.join(source_dir, item_name)
    
    # Enter each directory and look for photos/videos folders inside
    if os.path.isdir(item_path):
        try:
            for sub_item in os.listdir(item_path):
                sub_path = os.path.join(item_path, sub_item)
                if os.path.isdir(sub_path) and sub_item.lower() in ['photos', 'videos']:
                    shutil.rmtree(sub_path)
                    print(f"Deleted folder: {sub_path}")
        except Exception as e:
            print(f"Error accessing {item_path}: {e}")

print("Cleanup completed.")

Deleted folder: /Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_chats/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages/inbox/palihoviciangeldavid_557318612330154/photos
Deleted folder: /Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_chats/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages/inbox/angelgalaanu_899360558121329/videos
Deleted folder: /Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_chats/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages/inbox/golandiamama_323369479524009/videos
Deleted folder: /Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_chats/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages/inbox/golandiamama_323369479524009/photos
Deleted folder: /Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_chats/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

def process_instagram_messages(base_path):
    """
    Process Instagram message files into a pandas DataFrame.
    
    Args:
        base_path: Path to the inbox folder containing chat directories
    
    Returns:
        pandas.DataFrame with columns: message, sender, time, chat_id, chat_name
    """
    all_messages = []
    
    # Iterate through each chat folder
    for chat_folder in os.listdir(base_path):
        chat_path = os.path.join(base_path, chat_folder)
        
        if not os.path.isdir(chat_path):
            continue
            
        # Find all message files in the chat folder
        message_files = [f for f in os.listdir(chat_path) if f.startswith('message_') and f.endswith('.json')]
        
        for message_file in message_files:
            file_path = os.path.join(chat_path, message_file)
            
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                chat_name = data.get('title', chat_folder)
                chat_id = chat_folder
                
                # Process each message
                for msg in data.get('messages', []):
                    # Convert timestamp from milliseconds to datetime
                    timestamp = datetime.fromtimestamp(msg['timestamp_ms'] / 1000)
                    
                    message_data = {
                        'message': msg.get('content', ''),
                        'sender': msg.get('sender_name', ''),
                        'time': timestamp,
                        'chat_id': chat_id,
                        'chat_name': chat_name
                    }
                    all_messages.append(message_data)
    
    # Create DataFrame and sort by time
    df = pd.DataFrame(all_messages)
    df = df.sort_values('time').reset_index(drop=True)
    
    return df

# Usage
base_path = "/Users/pariidan/Documents/python_projects/RAG_Langchain/data/instagram_data/instagram-pariidan-2025-08-06-FS4vNf61/your_instagram_activity/messages/inbox"
messages_df = process_instagram_messages(base_path)


In [16]:
messages_df

,message,sender,time,chat_id,chat_name
0,Rofl,Instagram user,2017-09-21 21:28:53.223,instagramuser_537292367671997,Instagram user
1,Vad ca iai dat follow lu emy lungu,Instagram user,2017-09-21 21:37:06.128,instagramuser_537292367671997,Instagram user
2,â¤ï¸,Instagram user,2017-09-21 21:37:28.969,instagramuser_537292367671997,Instagram user
3,Emm,Dan Parii,2017-09-21 21:41:32.892,instagramuser_537292367671997,Instagram user
4,Nu,Dan Parii,2017-09-21 21:41:34.343,instagramuser_537292367671997,Instagram user
...,...,...,...,...,...
131004,Slabo sati iai,Dan,2025-08-04 20:56:39.964,dan_1212277636837264,Dan
131005,Dan sent an attachment.,Dan,2025-08-04 21:04:39.659,dan_1212277636837264,Dan
131006,Dan sent an attachment.,Dan,2025-08-04 21:09:23.721,dan_1212277636837264,Dan
131007,yoyo sent an attachment.,yoyo (Bogdan),2025-08-05 08:48:22.681,nidosorbo_25786883254290535,nido sorbo


In [ ]:
def create_chat_summary(df):
    """
    Create a summary DataFrame with chat metadata.
    
    Args:
        df: DataFrame from process_instagram_messages()
    
    Returns:
        pandas.DataFrame with columns: chat_id, chat_name, is_groupchat, participants
    """
    chat_summary = df.groupby(['chat_id', 'chat_name']).agg({
        'sender': lambda x: list(x.unique())
    }).reset_index()
    
    chat_summary['participants'] = chat_summary['sender']
    chat_summary['is_groupchat'] = chat_summary['participants'].apply(lambda x: len(x) > 2)
    chat_summary = chat_summary.drop('sender', axis=1)
    
    return chat_summary

chat_summary_df = create_chat_summary(messages_df)


In [18]:

def add_language_classification(messages_df, chat_summary_df, batch_size=500):

    from transformers import pipeline
    from tqdm import tqdm
    
    # Initialize classifier
    classifier = pipeline("text-classification",
                         model="qanastek/51-languages-classifier",
                         truncation=True,
                         max_length=512)
    
    language_map = {
        'ro-RO': 'ro',
        'en-US': 'en',
    }
    
    def classify_language(text):
        if not text or pd.isna(text) or text.strip() == '':
            return 'unknown'
        res = classifier(text)[0]
        if res['score'] > 0.5:
            label = language_map.get(res['label'], res['label'])
            return label
        else:
            return 'unknown(possibly slang)'
    
    # Filter out empty messages
    valid_messages = messages_df[messages_df['message'].notna() & (messages_df['message'].str.strip() != '')]
    
    # Process in batches with progress bar
    languages = []
    total_batches = len(valid_messages) // batch_size + (1 if len(valid_messages) % batch_size != 0 else 0)
    
    print(f"Processing {len(valid_messages)} messages in {total_batches} batches of {batch_size}")
    
    for i in tqdm(range(0, len(valid_messages), batch_size), desc="Processing batches"):
        batch = valid_messages.iloc[i:i+batch_size]
        batch_languages = [classify_language(msg) for msg in batch['message']]
        languages.extend(batch_languages)
    
    # Create a copy of messages_df to avoid modifying original
    messages_df_copy = messages_df.copy()
    
    # Initialize all messages as 'unknown'
    messages_df_copy['language'] = 'unknown'
    
    # Update with classified languages for valid messages
    messages_df_copy.loc[valid_messages.index, 'language'] = languages
    
    # Calculate most common language per chat
    chat_languages = messages_df_copy.groupby('chat_id')['language'].apply(lambda x: x.mode().iloc[0])
    messages_df_copy['chat_language'] = messages_df_copy['chat_id'].map(chat_languages)
    
    # Add chat language to summary DataFrame
    chat_summary_df_copy = chat_summary_df.copy()
    chat_summary_df_copy['chat_language'] = chat_summary_df_copy['chat_id'].map(chat_languages)
    
    return messages_df_copy, chat_summary_df_copy

# Add language classification
messages_df_with_lang, chat_summary_df_with_lang = add_language_classification(messages_df, chat_summary_df, batch_size=500)

print("\nMessages DataFrame with language:")
print(messages_df_with_lang[['message', 'sender', 'language', 'chat_language']].head())

print("\nChat Summary DataFrame with language:")
print(chat_summary_df_with_lang)

Device set to use mps:0


Processing 117243 messages in 235 batches of 500


Processing batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 235/235 [1:06:00<00:00, 16.85s/it]


Messages DataFrame with language:
                              message          sender language chat_language
0                                Rofl  Instagram user    km-KH            ro
1  Vad ca iai dat follow lu emy lungu  Instagram user       ro            ro
2                              â¤ï¸  Instagram user    mn-MN            ro
3                                 Emm       Dan Parii    sv-SE            ro
4                                  Nu       Dan Parii    sv-SE            ro

Chat Summary DataFrame with language:
                           chat_id         chat_name  \
0                 1696960598361317              ð   
1              18_1907003319402185               18!   
2           18ani_2643423222390720            18 ani   
3        18mneuje_2741369452561842        18 mne uje   
4                24012812184989611              ð   
..                             ...               ...   
141   vaaalikkkkkk_536449434424222      vaaalikkkkkk   
142     valeoizm

In [ ]:
messages_df_with_lang.to_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/instagram_messages.csv')
chat_summary_df_with_lang.to_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/instagram_chat_info.csv')

- Tomorrow : Make it same format as WhatsApp so can Reprocess Easiliy
- Setup Barebons API App (lesgoo)

In [59]:
import pandas as pd

messages_df = pd.read_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/instagram_messages.csv',index_col=0)
chat_summary_df = pd.read_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/instagram_chat_info.csv')

In [60]:
# Try to decode the corrupted string
def fix_encoding(text):
    if isinstance(text, str):
        try:
            # First encode as latin1, then decode as utf-8
            return text.encode('latin1').decode('utf-8')
        except:
            return text
    return text

# Apply to the sender column
messages_df['sender'] = messages_df['sender'].apply(fix_encoding)
messages_df['chat_name'] = messages_df['chat_name'].apply(fix_encoding)
messages_df['message'] = messages_df['message'].apply(fix_encoding)

#### More Data Cleaning

In [ ]:
whats_app_messages = pd.read_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/whatsapp/whatsapp_chats.csv',index_col=0)

In [63]:
whats_app_messages['content'].apply(str).apply(len).sum()

np.int64(3825091)

In [64]:
messages_df['message'].apply(str).apply(len).sum()

np.int64(2836319)

In [ ]:
messages_df.dropna(inplace=True)
messages_df = messages_df.loc[~messages_df['message'].str.contains('sent an attachment.', na=False)]
messages_df.loc[messages_df['chat_name'].str.contains('Cristina Sîli'),'chat_name'] = 'Cristina'
messages_df.loc[messages_df['sender'].str.contains('Cristina Sîli'),'sender'] = 'Cristina' 
messages_df.loc[messages_df['sender'].str.contains('Dan Parii'),'sender'] = 'Me'
messages_df['is_groupchat'] = messages_df['chat_id'].map(messages_df.groupby('chat_id')['sender'].nunique()>2) 
messages_df['participants'] = messages_df.groupby('chat_id')['sender'].transform(lambda x: [list(x.unique())] * len(x))

In [ ]:
# Extract date and time first
messages_df['time'] = pd.to_datetime(messages_df['time'])
date_col = messages_df['time'].dt.date
time_col = messages_df['time'].dt.time

# Drop original and add new columns
messages_df = messages_df.drop('time', axis=1)
messages_df['date'] = date_col
messages_df['time'] = time_col

messages_df.drop(index=22321,inplace=True)
messages_df.drop(index=85270,inplace=True)
messages_df.drop(index=messages_df[(messages_df['chat_name']=='Teodor Lungu and Mihai') & (~messages_df['is_groupchat'])].index,inplace=True) # weird duplicate rows

In [116]:
# check languages
messages_df['chat_language'].value_counts()

chat_language
ro    115344
en       748
Name: count, dtype: int64

In [ ]:
### set separate chat names for anonymous users
instagram_messages_df = messages_df[messages_df['chat_name'] == 'Instagram user']

i = 1  
for chat_id in instagram_messages_df['chat_id'].unique():
    mask = messages_df['chat_id'] == chat_id
    messages_df.loc[mask, 'sender'] = messages_df.loc[mask, 'sender'].str.replace('Instagram user', f'Instagram user {i}')
    messages_df.loc[mask, 'chat_name'] = messages_df.loc[mask, 'chat_name'].str.replace('Instagram user', f'Instagram user {i}')
    
    i += 1

In [ ]:
# rename columns
messages_df.rename(columns={'message' : 'content',},inplace=True)
messages_df['content_len'] = messages_df['content'].apply(len)

In [38]:
import pandas as pd
messages_df = pd.read_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/instagram_messages.csv',index_col=0)
#messages_df.to_csv('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/processed_data/instagram_messages.csv')

In [39]:
from custom_textsplitter import CustomTextSplitter

splitter = CustomTextSplitter()
docs = splitter.split_messages(chunk_size=1000, message_df=messages_df,return_df=False)


In [40]:
from custom_textsplitter import CustomTextSplitter

chunk_df = splitter.split_messages(chunk_size=1000, message_df=messages_df,return_df=True)